### 실습

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 데이터 로드
X_full = pd.read_csv('data/train.csv', index_col='Id')
X_test_full = pd.read_csv('data/test.csv', index_col="Id")

# 결측 데이터 삭제 - 'SalePrice'
X_full.dropna(axis=0, subset=['SalePrice'], inplace=True)
y = X_full.SalePrice
X_full.drop(['SalePrice'], axis=1, inplace=True)

# 데이터 분리
X_train_full, X_valid_full, y_train, y_valid = train_test_split(
    X_full, y, train_size=0.8, test_size=0.2, random_state=0
)

categorical_cols = [col for col in X_train_full.columns
                    if X_train_full[col].nunique() < 10 and
                    X_train_full[col].dtype == 'object']

print(">>>" , categorical_cols)

# 범주형 컬럼 추출
numberical_cols = [col for col in X_train_full.columns
                   if X_train_full[col].dtype in ['int64', 'float64']]


my_cols = categorical_cols + numberical_cols
X_train = X_train_full[my_cols].copy()
X_valid = X_valid_full[my_cols].copy()
X_test = X_test_full[my_cols].copy()

X_train.head()


from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error


# 데이터 전처리

# 1) 숫자형 데이터
numberical_transformer = SimpleImputer(strategy='constant')

# 2) 범주형 데이터
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])


# 숫자형 + 범주형 묶기
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numberical_transformer, numberical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

# 모델 정의
model = RandomForestRegressor(n_estimators=100, random_state=0)

# 파이프라인 정의
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

# 모델 학습
clf.fit(X_train, y_train)

preds = clf.predict(X_valid)

print("MAE : ", mean_absolute_error(y_valid, preds))


# 테스트 데이터 검증
my_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

my_pipeline.fit(X_train, y_train)

preds_test = my_pipeline.predict(X_test)

#score = mean_absolute_error(y_valid, preds_test)
#print("MAE : " , score)


>>> ['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition']
MAE :  17614.81993150685
